In [237]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle
from torch.utils.data import Dataset, DataLoader, random_split

### 0. Prepare

In [2]:
with open('/content/drive/MyDrive/datasets/MIR/0_100.pkl', 'rb') as f:
    pkl0to100 = pickle.load(f)

with open('/content/drive/MyDrive/datasets/MIR/100_600.pkl', 'rb') as f:
    pkl100o600 = pickle.load(f)

with open('/content/drive/MyDrive/datasets/MIR/600_1100.pkl', 'rb') as f:
    pkl600to1100 = pickle.load(f)

with open('/content/drive/MyDrive/datasets/MIR/1100_1600.pkl', 'rb') as f:
    pkl1100to1600 = pickle.load(f)

with open('/content/drive/MyDrive/datasets/MIR/1600_2076.pkl', 'rb') as f:
    pkl1600to2076 = pickle.load(f)

In [3]:
mir_total = {}

pkl_list = [pkl0to100, pkl100o600, pkl600to1100, pkl1100to1600, pkl1600to2076]
for pkl in pkl_list:
    for key, value in pkl.items():
        mir_total[key] = value

In [4]:
with open('/content/drive/MyDrive/datasets/MIR/mir_total.pkl', 'wb') as f:
      pickle.dump(mir_total, f)

In [302]:
labels = np.load('/content/drive/MyDrive/datasets/MIR/final_final_result_by_Gemini.npy')

In [310]:
sums = np.nansum(labels, axis=0).astype('int')

In [110]:
labels2 = {}
for i, value in enumerate(labels):
    labels2[i] = value

In [112]:
with open('/content/drive/MyDrive/datasets/MIR/label_total.pkl', 'wb') as f:
      pickle.dump(labels2, f)

In [165]:
with open('/content/drive/MyDrive/datasets/MIR/mir_total.pkl', 'rb') as f:
    latents = pickle.load(f)

In [300]:
with open('/content/drive/MyDrive/datasets/MIR/label_total.pkl', 'rb') as f:
    labels = pickle.load(f)

In [301]:
labels

{0: array([0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0.,
        0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0.,
        0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1., 0., 1., 1.,
        0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.,
        0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 0., 0.,
        1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
        1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 0., 0., 0.,
        0

In [167]:
import math

outputs = []
for key, value in labels.items():
  for x in value:
    if math.isnan(x):
      outputs.append(key)
      break

In [168]:
len(outputs)

190

In [169]:
additive_x = {}
# key : music id
# value : tensor (x or y)

for i, num in enumerate(outputs):
  additive_x[num] = latents[num]
  del latents[num]
  del labels[num]

In [173]:
len(latents), len(labels), len(additive_x)

(1886, 1886, 190)

In [176]:
data_x = {}
data_y = {}
id_to_new_id = {}

for i, (key, value) in enumerate(latents.items()):
  id_to_new_id[key] = i
  data_x[i] = value
for i, (key, value) in enumerate(labels.items()):
  data_y[i] = value

In [179]:
with open('/content/drive/MyDrive/datasets/MIR/data_x.pkl', 'wb') as f:
      pickle.dump(data_x, f)

with open('/content/drive/MyDrive/datasets/MIR/data_y.pkl', 'wb') as f:
      pickle.dump(data_y, f)

with open('/content/drive/MyDrive/datasets/MIR/additive_x.pkl', 'wb') as f:
      pickle.dump(additive_x, f)

with open('/content/drive/MyDrive/datasets/MIR/id_to_new_id.pkl', 'wb') as f:
      pickle.dump(id_to_new_id, f)

### 1. Dataset

In [258]:
with open('/content/drive/MyDrive/datasets/MIR/data_x.pkl', 'rb') as f:
    data_x = pickle.load(f)

In [259]:
with open('/content/drive/MyDrive/datasets/MIR/data_y.pkl', 'rb') as f:
    data_y = pickle.load(f)

In [274]:
class AudioDataset(Dataset):
    def __init__(self, latents, labels):
        self.latents = latents
        self.labels = labels

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        x_latent = self.latents[idx]
        y_label = self.labels[idx]

        return torch.tensor(x_latent, dtype=torch.float32), torch.tensor(y_label, dtype=torch.float32)

In [275]:
dataset = AudioDataset(data_x, data_y)

In [276]:
latent, label = dataset.__getitem__(569)

print(latent.size())
print(label.size())

torch.Size([13, 4800])
torch.Size([305])


### 2. LSTM Classifier

In [279]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # LSTM을 통과한 후, 마지막 타임 스텝의 은닉 상태를 사용
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))

        # 마지막 타임 스텝의 출력을 분류기에 전달
        out = self.fc(out[:, -1, :])
        return out

# 모델 초기화
input_dim = 4800
hidden_dim = 1024
num_layers = 12
num_classes = 305

model = LSTMClassifier(input_dim, hidden_dim, num_layers, num_classes)

### 3. Train

In [280]:
torch.cuda.empty_cache()

In [297]:
epochs = 100
learning_rate = 0.05

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [298]:
best_loss = float('inf')

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    running_loss_list = []
    print(f"---------------------Epoch {epoch+1}-----------------------")
    for i in range(len(train_dataset)):
        data, target = train_dataset[i]
        data, target = data.unsqueeze(0).to(device), target.unsqueeze(0).to(device)

        optimizer.zero_grad()

        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        running_loss_list.append(loss.item())

        if i%500 == 0:
            print(f'data {i+1} is done')

    # 에포크 평균 손실 계산
    epoch_loss = running_loss / len(train_dataset)
    print(f'Loss of Epoch {epoch+1}: {epoch_loss}')

    # 현재 에포크의 손실이 이전에 기록된 최소 손실보다 작은 경우, 모델 저장
    if epoch_loss < best_loss:
        best_loss = epoch_loss


---------------------Epoch 1-----------------------
data 1 is done
data 501 is done
data 1001 is done
data 1501 is done
Loss of Epoch 1: 204.91664757399724
---------------------Epoch 2-----------------------
data 1 is done
data 501 is done
data 1001 is done
data 1501 is done
Loss of Epoch 2: 204.81221098672174
---------------------Epoch 3-----------------------
data 1 is done
data 501 is done
data 1001 is done
data 1501 is done
Loss of Epoch 3: 204.7411982374419
---------------------Epoch 4-----------------------
data 1 is done
data 501 is done
data 1001 is done
data 1501 is done
Loss of Epoch 4: 204.70696403208714
---------------------Epoch 5-----------------------
data 1 is done
data 501 is done
data 1001 is done
data 1501 is done
Loss of Epoch 5: 204.6760878101268
---------------------Epoch 6-----------------------
data 1 is done
data 501 is done
data 1001 is done
data 1501 is done
Loss of Epoch 6: 204.64869915016135
---------------------Epoch 7-----------------------
data 1 is done

In [296]:
answer = test_dataset[0][1]
input = test_dataset[0][0]

output = model(input.unsqueeze(0).to(device))
pred = nn.functional.softmax(output)


top10_preds = torch.topk(pred, 10)
print(top10_preds.indices)
answer_indices = (answer == 1).nonzero(as_tuple=True)[0]
print(answer_indices)

tensor([[128, 182, 127,  21, 234, 270, 233,  70, 157, 273]], device='cuda:0')
tensor([ 10,  11,  25,  35,  40,  49,  80,  85,  86,  90,  91, 101, 104, 105,
        112, 132, 133, 139, 141, 157, 175, 182, 185, 189, 192, 193, 196, 257,
        262, 267, 270, 290])


<ipython-input-296-cff9f77b8bc6>:5: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  pred = nn.functional.softmax(output)


In [299]:
torch.save({
            'epoch': epoch,
            'leraning rate': 0.001,
            'hidden_dim' : hidden_dim,
            'num_layers' : num_layers,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, '/content/drive/MyDrive/model/total_model1.pth')